# SUPRA-JEPA sur Google Colab

Notebook pour preparer les donnees, pre-entrainer SUPRA-JEPA, puis lancer le fine-tuning binaire supraconducteur / non-supraconducteur.

Avant de lancer : dans Colab, active `Runtime > Change runtime type > GPU` si disponible.

In [ ]:
import os, sys, json, subprocess, textwrap
from pathlib import Path

print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch pas encore importe:', exc)

## 1. Installer les dependances

Colab contient deja PyTorch. On installe surtout `matminer`, `mp-api`, `pymatgen`, `spglib` et `scikit-learn`.

In [ ]:
!pip -q install matminer mp-api pymatgen spglib scikit-learn

## 2. Charger le projet

Par defaut, le notebook clone ton repo GitHub dans `/content/SUPRACONDUCTOR-JEPA`.

In [ ]:
REPO_URL = 'https://github.com/Baptistecaille/SUPRACONDUCTOR-JEPA.git'
PROJECT_DIR = Path('/content/SUPRACONDUCTOR-JEPA')

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Projet introuvable: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', Path.cwd())
!ls -la

## 3. Parametres Colab

Ces valeurs sont petites pour valider le pipeline. Augmente-les ensuite pour un vrai run.

In [ ]:
MAX_SC = 100
MAX_NONSC = 100

PRETRAIN_EPOCHS = 3
FINETUNE_EPOCHS = 5
BATCH_SIZE_PRETRAIN = 32
BATCH_SIZE_FINETUNE = 32

print({
    'MAX_SC': MAX_SC,
    'MAX_NONSC': MAX_NONSC,
    'PRETRAIN_EPOCHS': PRETRAIN_EPOCHS,
    'FINETUNE_EPOCHS': FINETUNE_EPOCHS,
})

## 4. Preparer les donnees

Il faut une cle API Materials Project. Elle est demandee de maniere masquee.

In [ ]:
from getpass import getpass

mp_api_key = getpass('Materials Project API key: ')
cmd = [
    sys.executable, '-m', 'scripts.prepare_data',
    '--mp-api-key', mp_api_key,
    '--max-sc', str(MAX_SC),
    '--max-nonsc', str(MAX_NONSC),
]
subprocess.run(cmd, check=True)

In [ ]:
from data import load_supercon_dataset

structures, labels = load_supercon_dataset('data')
print('Total:', len(labels))
print('SC:', sum(labels))
print('non-SC:', len(labels) - sum(labels))

## 5. Pretraining JEPA

On appelle les fonctions Python directement pour surcharger les hyperparametres sans modifier les fichiers du repo.

In [ ]:
import torch
from torch.utils.data import DataLoader

from config import Config
from data import CrystalDataset
from model import SupraJEPA
from pretrain import pretrain

cfg = Config()
cfg.epochs_pretrain = PRETRAIN_EPOCHS
cfg.batch_size_pretrain = BATCH_SIZE_PRETRAIN
cfg.warmup_steps = 10
cfg.data_dir = 'data'

torch.manual_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

pretrain_ds = CrystalDataset(structures, [-1] * len(structures), cfg.max_atoms, cfg.mask_ratio)
pretrain_loader = DataLoader(pretrain_ds, batch_size=cfg.batch_size_pretrain, shuffle=True, num_workers=2)

model = SupraJEPA(cfg)
print('Parametres entrainables:', sum(p.numel() for p in model.parameters() if p.requires_grad))
model = pretrain(model, pretrain_loader, cfg, device)

## 6. Fine-tuning classification

In [ ]:
from data import make_dataloaders
from finetune import finetune

cfg.epochs_finetune = FINETUNE_EPOCHS
cfg.batch_size_finetune = BATCH_SIZE_FINETUNE

train_dl, val_dl, test_dl = make_dataloaders(structures, labels, cfg)

train_labels = train_dl.dataset.labels

model = SupraJEPA(cfg)
model = finetune(model, train_dl, val_dl, test_dl, train_labels, cfg, device)

## 7. Sauvegarder les artefacts

In [ ]:
!ls -lh checkpoints data

# Optionnel: sauvegarder dans Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/SUPRA-JEPA
# !cp -r checkpoints data /content/drive/MyDrive/SUPRA-JEPA/

## Notes

- Le matching SuperCon -> Materials Project utilise des formules candidates exactes puis arrondies, car beaucoup de compositions experimentales sont dopees ou non stoechiometriques.
- Pour un vrai entrainement, augmente `MAX_SC`, `MAX_NONSC`, `PRETRAIN_EPOCHS` et `FINETUNE_EPOCHS`.
- Sur CPU, garde des petits batchs. Sur GPU Colab, tu peux monter progressivement.